# VQA + TTS baseline

Photograph or upload one package. The HKBU vision model returns only its visible medication name and strength; gTTS reads those fields aloud. No reference list, automatic matching, health chat or medical advice is supplied. The participant compares the response with the separate reference list and confirms their selection. No actual medication is taken.


## 1. Install dependencies
Windows, macOS and Linux use the same Python packages. No Tesseract or system OCR executable is required. Restart Jupyter/kernel if newly installed widget extensions are not loaded.


In [ ]:
%pip install "anywidget>=0.9,<1" "ipywidgets>=8,<9" "Pillow>=10,<13" "gTTS>=2.5,<3" "requests>=2.31,<3"


## 2. API configuration

Enter the HKBU API key below once and rerun this cell after editing. Use the same model and API version as the full system. The key is kept in Python, not sent to frontend widget state or printed. **A key typed here is saved in the notebook source: replace it with the placeholder before sharing or uploading this notebook.**


In [ ]:
API_KEY = "YOUR_HKBU_API_KEY"
MODEL = "gpt-4.1"
API_VERSION = "2025-04-01-preview"

if not API_KEY.strip() or API_KEY.startswith("YOUR_"):
    raise ValueError("Enter your HKBU API key above, then rerun this cell.")
print("Configuration ready. Model:", MODEL)


## 3. Open the reader

Open the camera, capture a label, listen to the name and strength and compare with your separately supplied list. You may retake, upload, replay or cancel. Click **I selected this box — finish task** when you have chosen. Selection correctness is scored by the evaluator, not this tool.


In [ ]:
import sys
import importlib
from pathlib import Path
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "baselines" / "vqa_tts", Path.cwd() / "vqa_tts"]
BASELINE_DIR = next((p for p in candidates if (p / "vqa_baseline.py").is_file()), None)
if BASELINE_DIR is None:
    raise FileNotFoundError("Open this notebook from baselines/vqa_tts or from the repository root.")
if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))
if "reader" in globals():
    reader.close()
import vqa_baseline
importlib.reload(vqa_baseline)
reader = vqa_baseline.VQALabelReader(API_KEY, model=MODEL, api_version=API_VERSION, folder=BASELINE_DIR / "vqa_records")
display(reader)


## 4. Evaluation records

`vqa_records/tasks.json` stores task and photo-attempt details; `vqa_records/summary.csv` stores summaries. Timing starts at first photo submission and ends at accepted selection confirmation, including network, model, TTS, listening and retakes. Initial searching/camera opening is excluded, matching the current MediConfirm website metric. Completed does not mean correct.

Button clicks are counted separately from audio-play events. Click rate is undefined. No photos, recordings, API keys or model text are logged. Records are separate from the main system. Use one active widget per records folder. Actual model quality, camera behavior and audio require local testing.


In [ ]:
import csv
path = BASELINE_DIR / "vqa_records" / "summary.csv"
if path.exists():
    with path.open(newline="") as file:
        for row in csv.DictReader(file):
            print(row)
    print("CSV:", path)
else:
    print("No records yet.")
